# <a id='toc1_'></a>[***Notebook pédagogique n°6 : TD de révision - tune-model-saver***](#toc0_)

# <a id='toc2_'></a>[Auteurs](#toc0_)
- [Claude CALIGARIS](claude.caligaris@edf.fr)
- [François CHANEL](francois.chanel@edf.fr)
- [Olivier COUDRAY](olivier.coudray@edf.fr)
- [Léon DO CASTELO](leon.do-castelo@edf.fr)
- [Paul GRENTE](paul.grente@edf.fr)
- [Inès OUKID](ines.oukid@edf.fr)

# <a id='toc3_'></a>[Notions abordées :](#toc0_)
- programmation objet (déclaration de classes)
- BaseModels pydantic
- héritage
- Enum
- logging

# <a id='toc4_'></a>[Sommaire](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [***Notebook pédagogique n°6 : TD de révision - tune-model-saver***](#toc1_)    
- [Auteurs](#toc2_)    
- [Notions abordées :](#toc3_)    
- [Sommaire](#toc4_)    
- [Contexte](#toc5_)    
- [Imports](#toc6_)    
- [Toy DataSet pour ce TD](#toc7_)    
- [$1^{ère}$ partie : coder une classe héritée de `GridSearchCV`](#toc8_)    
- [$2^{ème}$ partie : enumérer les grid-searchs](#toc9_)    
- [$3^{ème}$ partie  : mécanismes de sauvegarde](#toc10_)    
  - [Opérateur de sauvegarde](#toc10_1_)    
    - [Formation des connecteurs S3](#toc10_1_1_)    
    - [constantes utiles](#toc10_1_2_)    
  - [Héritage, spécialisation, et Factory](#toc10_2_)    
    - [Spécialisation](#toc10_2_1_)    
    - [Factory pour générer des opérateurs spécialisés sans jamais se tromper !](#toc10_2_2_)    
  - [Représentants des fichiers à sauvegarder](#toc10_3_)    
    - [classe représentant un fichier quelconque](#toc10_3_1_)    
    - [TunedModelFiles déterminés](#toc10_3_2_)    
  - [Injection du mécanisme de sauvegarde dans la classe `RegressorGridSearcher`](#toc10_4_)    
  - [Injection d'un mécanisme de _lecture_](#toc10_5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc5_'></a>[Contexte](#toc0_)

Dans le cadre d'un UseCase (le UC172 pour ne pas le nommer), il a été nécessaire de tester plusieurs types de régresseurs. Sur chaque régresseur, on effectuait une recherche d'hyper-paramètres optimaux exhaustive, via l'opérateur `GridSearchCV` de scikit-learn fitté sur les données train. Une fois la recherche terminée, on souhaite pouvoir sauvegarder facilement l'ensemble des attributs intéressants : 
- les meilleurs hyper-paramètres recensés (attribut `.best_params_` de l'instance `GridSearchCV`)
- le fichier global de scores des fits par cross-validation (attribut `.cv_results_`)
- les scores obtenus sur train puis sur test
- l'estimateur optimisé lui-même

On souhaitait dès lors se doter d'un moyen simple de : 
- définir ce qui constituait les "meilleurs morceaux" d'un grid entraîné, à sauvegarder
- définir une convention de nommage pour les sauvegarder
- se doter des méthodes d'upload sur S3

![image](./docs/images/fine-tuned-model-parts.png)

# <a id='toc6_'></a>[Imports](#toc0_)

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import (
    train_test_split, 
    GridSearchCV,
    learning_curve,
    LearningCurveDisplay,
)
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, make_scorer

import numpy as np
import pandas as pd
from dataclasses import dataclass
from pydantic import BaseModel, Field, PrivateAttr
from typing import Self, Callable, Any, Literal, TypeVar, Type
from enum import Enum
from copy import deepcopy
from loguru import logger
from matplotlib import pyplot as plt

# <a id='toc7_'></a>[Toy DataSet pour ce TD](#toc0_)

On crée un générateur de toy-data-sets pour alimenter cet exemple

In [ ]:
N_SAMPLES__DEFAULT: int = 200
N_FEATURES__DEFAULT: int = 5
NOISE__DEFAULT: float = 5.0
RANDOM_STATE__DEFAULT: int = 42
TRAIN_TEST_SPLIT_RATIO__DEFAULT: float = 0.25
SAMPLE_WEIGHT__DEFAULT: float = 10.0
SAMPLE_WEIGHT_RATIO__DEFAULT: float = 0.3

In [ ]:
@dataclass(frozen=True, kw_only=True)
class DataGenerator:
    """
    A simple class to generate a synthetic regression dataset.

    Attributes:
        n_samples (int): Number of samples.
        n_features (int): Number of features.
        noise (float): Standard deviation of the gaussian noise applied to the output.
        random_state (int): Seed for random number generation.
    """
    n_samples: int = N_SAMPLES__DEFAULT
    n_features: int = N_FEATURES__DEFAULT
    noise: float = NOISE__DEFAULT
    random_state: int = RANDOM_STATE__DEFAULT

    def generate(self):
        """
        Generate the regression dataset.

        Returns:
            X (ndarray): Feature matrix.
            y (ndarray): Target vector.
        """
        X, y = make_regression(
            n_samples=self.n_samples,
            n_features=self.n_features,
            noise=self.noise,
            random_state=self.random_state,
        )
        return X, y

    def get_train_test_split(
        self, 
        test_size: float = TRAIN_TEST_SPLIT_RATIO__DEFAULT,
        sample_weight: float = SAMPLE_WEIGHT__DEFAULT,
        sample_weight_ratio: float = SAMPLE_WEIGHT_RATIO__DEFAULT,
    ):
        """
        Generate the dataset, split it into training and testing sets, and generate sample weights for the training set.

        25% of the training samples will be assigned an external weight of 10.0, while the rest receive a default weight of 1.0.

        Args:
            test_size (float): Proportion of the dataset to include in the test split.

        Returns:
            X_train, X_test, y_train, y_test, sample_weights: Splitted dataset arrays and corresponding sample weights for training set.
        """
        # Generate the full dataset
        X, y = self.generate()
        # Split the dataset into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )

        # Initialize all training sample weights to 1.0
        sample_weights = np.ones_like(y_train, dtype=float)
        n_train = y_train.shape[0]
        # Determine the number of samples to assign the external weight (25% of training samples)
        n_weighted = int(sample_weight_ratio * n_train)

        # Create a random generator using the provided seed for reproducibility
        rng = np.random.RandomState(self.random_state)
        # Randomly select indices for which the sample weight will be set to 10.0
        weighted_indices = rng.choice(n_train, size=n_weighted, replace=False)
        sample_weights[weighted_indices] = sample_weight

        return X_train, X_test, y_train, y_test, sample_weights

In [ ]:
data_generator = DataGenerator()

In [ ]:
(
    X_train, 
    X_test, 
    y_train, 
    y_test, 
    sample_weights,
) = data_generator.get_train_test_split()

X = np.concatenate([X_train, X_test])
y = np.concatenate([y_train, y_test])

In [ ]:
scorer__rmse = make_scorer(
    score_func=root_mean_squared_error,
    greater_is_better=False,
)

# <a id='toc8_'></a>[$1^{ère}$ partie : coder une classe héritée de `GridSearchCV`](#toc0_)

Créer une classe qui **hérite** de la classe `GridSearchCV` $^{(+)}$, et, en plus de l'héritage qui lui permet de faire tout ce que fait GridSearch, lui rajoute : 
- une property pour renvoyer **le nom de la classe d'estimateur sous-jacente**
- une adaptation de la méthode `.fit()` pour gérer sans effort le cas "compatible avec sample-weight ou non"
- des méthodes pour calculer le score sur le jeu train (X_train et y_train) et sur test (X_test et y_test) $^{(*)}$
- une méthode pour calculer la learning-curve sur l'ensemble des données (X, y) $^{(o)}$
- une méthode pour afficher la learning curve $^{(o)}$
- des attributs et properties pour contenir (enregistrer) ces 2 score et pouvoir rappeler les 2 à la fois d'un seul tenant
- une méthode de sauvegarde sur S3  $^{(*)}$
- une méthode pour codifier de façon explicite la représentation en str de la classe (afin qu'elle rappelle à minima quelle est la classe d'estimateur sous-jacent)

$^{(*)}$ Pour l'instant il n'est pas nécessaire que ces méthodes additionnelles fassent autre chose que ***pass*** : leur présence permet juste d'avoir tout de suite l'architecture générale.

Ensuite, entraînez votre classe sur X_train et y_train, et constatez l'accès aux attributs attendus post-fits.  

Pas de stress : on va faire ça **par étapes progressives !**
_____________

Pour rappel, votre classe va **hériter** de la classe `GridSearchCV` $^{(+)}$, donc elle va bénéficier : 
- de tous ses attributs
- de toutes ses méthodes  
$\implies$ il est essentiel que vous les connaissiez, afin de savoir : 
  - ce qui est d'ores et déjà à votre disposition
  - ce qu'attendent les méthodes déjà existantes

$^{(+)}$ lien vers la [documentation de la classe GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)  
$^{(o)}$ lien vers la [documentation de la learning-curve](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html) ainsi que [sur le displayer](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LearningCurveDisplay.html#sklearn.model_selection.LearningCurveDisplay)

> Etape 1 : créer une classe qui hérite de GridSearchCV, et qui prépare dès instanciation les attributs qui réceptionneront les scores sur train et test

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> Etape 2 : maintenant ajoutez-y de quoi : 
> - accéder au nom de l'estimateur sous-jacent
> - accéder à un résumé des scores

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> Etape 3 : ajoutez-y de quoi maîtriser sa représentation console, en indiquant a minima la nature de l'estimateur sous-jacent

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> Etape 4 : maintenant ajoutez des méthodes **qui ne font rien** (ie : qui font ***pass***) pour calculer les scores qui nous intéressent.  
> Afin d'éviter la duplication de code, on fera : 
> - une méthode **interne à la classe** qui compute le score sur X, y quelconque
> - les deux méthodes qui font ça sur train et test

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> Etape 5 : maintenant ajoutez une méthode pour sauvegarder sur S3 (elle fait ***pass*** à ce stade)

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> Procéder à un essai de cette classe, avec la grille de paramètres suivante, c'est à dire : 
> - instanciez la
> - entraînez là 
> - procédez à quelques visualisations / vérifications

In [ ]:
GRID_PARAMS__RANDOM_FOREST_REGRESSOR: dict = {
    "n_estimators": [
        5,
        50,
        100,
    ], 
    "max_depth": [
        1,
        10,
        20,
    ], 
    "min_samples_leaf": [
        1,
        2,
        4,
    ],
}

REGRESSOR_GRID_SEARCHER__COMMON_KWARGS: dict[str, Any] = dict(
    cv=5,
    scoring=scorer__rmse,
)

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)

> Utilisez les méthodes natives de `GridSearchCV` pour entraîner votre instance

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)

> Etape 6 :  Maintenant, mis à part la méthode de sauvegarde sur S3, codez explicitement les autres méthodes ajoutées.  <br><br>
> $[IMPORTANT]$ Pour cela il est nécessaire d'exploiter des méthodes **natives** de la classe `GridSearchCV`, dont votre nouvelle classe a **hérité**. Notamment, dans une instance de `GridSearchCV`, entraînée, la méthode `.score(X, y)` permet de calculer le score via le scorer passé en argument, avec le bon signe...

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> testez la classe actuelle sur le jeu d'hyper-paramètres et de données d'entraînement

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)

> Etape 7 : Ajoutez maintenant de quoi calculer la learning-curve, et stocker les résultats, puis y réaccéder par la suite facilement

In [ ]:
LEARNING_CURVE__TRAIN_SIZES: list[float] = list(map(
    lambda q: q/10,
    range(1, 11)
))

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)
grid.fit(X_train, y_train)
grid.compute_score_on_train(X_train, y_train)
grid.compute_score_on_test(X_test, y_test)
grid.compute_learning_curve(X, y);

> Etape 7-bis : Maintenant ajoutez une méthode pour afficher une illustration de la learning-curve (utiliser la classe `LearningCurveDisplay`)

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)
grid.fit(X_train, y_train)
grid.compute_score_on_train(X_train, y_train)
grid.compute_score_on_test(X_test, y_test)
grid.compute_learning_curve(X, y);

In [ ]:
grid.display_learning_curve()

> Etape 8 : on souhaite maintenant que la méthode `.fit()` bénéficie de la prise en compte des sample-weights, **lorsque c'est possible** (c'est à dire compatible avec l'estimateur sous-jacent) : 
> - comment codifier la compatibilité ou non ?
> - comment l'implémenter ensuite ?

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(
        self, 
        accepts_sample_weight: bool,
        **data,
    ):
        super().__init__(**data)
        ...

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    accepts_sample_weight=True,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)
grid.fit(X_train, y_train, sample_weight=sample_weights)
grid.compute_score_on_train(X_train, y_train)
grid.compute_score_on_test(X_test, y_test);

> Finalement, on juge que c'est dommage de ne pas profiter du moment où on fit le modèle sur (X_train, y_train) pour également calculer le score sur (X_train, y_train) ...   
> Modifiez la méthode `fit()` afin d'y loger ce calcul

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(
        self, 
        accepts_sample_weight: bool,
        **data,
    ):
        super().__init__(**data)
        ...

> Maintenant, incluez dans la classe tous les éléments qui seront ensuite à sauvegarder ensuite sur s3, de façon explicite. Pour rappel on veut pouvoir accéder, une fois le .fit() réalisé, à : 
> - un résumé des scores sur train et sur test (déjà réalisé grâce à la property scores)
> - aux données issues du calcul de la learning-curve (déjà réalisé grâce à la property learning_curve_data)
> - aux meilleurs hyper-paramètres identifiés via le gridsearch (attribut `.best_params_` sur la classe `GridSearchCV`), via `.best_hyper_parameters`
> - au résumé des scores obtenus sur tous les fits menés (attribut `.cv_results_` sur la classe `GridSearchCV`), via `.cv_results`
> - au meilleur estimateur résultant (attribut `.best_estimator_` sur la classe `GridSearchCV`), via `.best_estimator_fitted`
>
> La question est un peu "scolaire" ;-) au sens où **OUI**, pour les 3 derniers cas, ça revient juste à "donner un nouveau nom" artificiellement, en créeant une property qui se contente d'accéder à un attribut

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(
        self, 
        accepts_sample_weight: bool,
        **data,
    ):
        super().__init__(**data)
        ...

In [ ]:
grid = RegressorGridSearcher(
    estimator=RandomForestRegressor(),
    param_grid=GRID_PARAMS__RANDOM_FOREST_REGRESSOR,
    accepts_sample_weight=True,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS,
)
grid.fit(X_train, y_train)
grid.compute_score_on_test(X_test, y_test)
grid.compute_learning_curve(X=X, y=y);

# <a id='toc9_'></a>[$2^{ème}$ partie : enumérer les grid-searchs](#toc0_)

On se propose de ne pas former d'objets `RegressorGridSearcher` à la volée, mais plutôt **d'encadrer leur formation plus rigoureusement**, dans le cadre d'un Enum qui aura pour rôle de définir : 
- quels types de régresseurs seront à tester
- avec quelle grille de paramètre exploratoire

Ainsi, dans un script ultérieur qui les appelle tous pour les faire travailler, il suffira d'itérer dessus

> A partir des 3 jeux d'hyper-paramètres et des 3 regresseurs non-fittés fournis, générer un Enum qui comporte 3 membres et qui fournit des objets `RegressorGridSearcher`

In [ ]:
GRID_PARAMS__RANDOM_FOREST_REGRESSOR: dict = {
    "n_estimators": [
        5,
        50,
        100,
    ], 
    "max_depth": [
        1,
        10,
        20,
    ], 
    "min_samples_leaf": [
        1,
        2,
        4,
    ],
}

In [ ]:
GRID_PARAMS__ELASTIC_NET = {
    "alpha": [
        0.1,
        1,
        100,
    ],
    "l1_ratio": [
        0.1,
        0.5,
        0.9,
    ],
    "tol": [
        1e-4,
        1e-1,
    ],
}

In [ ]:
GRID_PARAMS__K_NEIGHBORS_REGRESSOR: dict = {
    "n_neighbors": [
        2,
        5,
        20,
    ],
    "weights": [
        "uniform",
        "distance",
    ],
    "leaf_size": [
        3,
        5,
        50,
    ],
}

In [ ]:
ACCEPTS_SAMPLE_WEIGHT__RANDOM_FOREST_REGRESSOR: bool = True
ACCEPTS_SAMPLE_WEIGHT__ELASTIC_NET: bool = True
ACCEPTS_SAMPLE_WEIGHT__K_NEIGHBORS_REGRESSOR: bool = False

In [ ]:
UNFITTED__RANDOM_FOREST_REGRESSOR: RandomForestRegressor = RandomForestRegressor()
UNFITTED__ELASTIC_NET: ElasticNet = ElasticNet()
UNFITTED__K_NEIGHBORS_REGRESSOR: KNeighborsRegressor = KNeighborsRegressor()

In [ ]:
REGRESSOR_GRID_SEARCHER__COMMON_KWARGS: dict[str, Any] = dict(
    cv=5,
    scoring=scorer__rmse,
)

$\Downarrow$ on forme dès lors les kwargs "complet" pour chaque famille de régresseur

In [ ]:
RANDOM_FOREST_REGRESSOR__FULL_KWARGS: dict = dict(
    estimator=deepcopy(UNFITTED__RANDOM_FOREST_REGRESSOR),
    param_grid=deepcopy(GRID_PARAMS__RANDOM_FOREST_REGRESSOR),
    accepts_sample_weight=ACCEPTS_SAMPLE_WEIGHT__RANDOM_FOREST_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS
)

ELASTIC_NET__FULL_KWARGS: dict = dict(
    estimator=deepcopy(UNFITTED__ELASTIC_NET),
    param_grid=deepcopy(GRID_PARAMS__ELASTIC_NET),
    accepts_sample_weight=ACCEPTS_SAMPLE_WEIGHT__ELASTIC_NET,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS
)

K_NEIGHBORS_REGRESSOR__FULL_KWARGS: dict = dict(
    estimator=deepcopy(UNFITTED__K_NEIGHBORS_REGRESSOR),
    param_grid=deepcopy(GRID_PARAMS__K_NEIGHBORS_REGRESSOR),
    accepts_sample_weight=ACCEPTS_SAMPLE_WEIGHT__K_NEIGHBORS_REGRESSOR,
    **REGRESSOR_GRID_SEARCHER__COMMON_KWARGS
)

In [ ]:
class RegressorGridSearchers(Enum):
    """simple Enum to describe all RegressorGridSearcher formed"""
    ...

> Entraînez tous les modèles, en itérant sur l'Enum, et en loggant à chaque étape ce qui se passe

In [ ]:
for grid_searcher in RegressorGridSearchers:
    ...

# <a id='toc10_'></a>[$3^{ème}$ partie  : mécanismes de sauvegarde](#toc0_)

On se propose maintenant de coder tout un mécanisme de sauvegarde de ce qui nous intéresse, sur S3, tous entraînements réalisés, afin de doter la classe `RegressorGridSearcher` d'une méthode de sauvegarde toute simple qui fait tout le travail pour nous d'un coup, en sauvegardant les bons objets au bon endroit 

## <a id='toc10_1_'></a>[Opérateur de sauvegarde](#toc0_)

> On veut se doter d'un opérateur capable de lire et d'écrire sur S3 à un endroit bien précis, pourvu qu'on lui ait indiqué comment faire

### <a id='toc10_1_1_'></a>[Formation des connecteurs S3](#toc0_)

In [ ]:
from pathlib import Path
from at_code_quality.constants import SECRET_PATH

In [ ]:
from daptools import (
    PolarsDataEngine,
    PandasDataEngine,
    AWSSettings,
)

In [ ]:
PDE = PandasDataEngine()
PLE = PolarsDataEngine()
engines = [PDE, PLE]
aws_settings = AWSSettings(config_file_path=SECRET_PATH)

for engine in engines:
    engine.connect_to_s3(settings=aws_settings, name="s3")

### <a id='toc10_1_2_'></a>[constantes utiles](#toc0_)

In [ ]:
S3_ROOT = r"s3:/"
FACTORY_BUCKET_ROOT: str = r"bkt-pud-uc"
CODE_QUALITY_FOLDER_ROOT: str = r"at001-quc"
TUTORIAL_FOLDER__NAME: str = "td__tuned_model_saver"

TUTORIAL_FOLDER__PATH: str = r"/".join([
    S3_ROOT,
    FACTORY_BUCKET_ROOT,
    CODE_QUALITY_FOLDER_ROOT,
    TUTORIAL_FOLDER__NAME,
])

In [ ]:
TUTORIAL_FOLDER__PATH

In [ ]:
MODELS_SELECTION__BEST_HYPER_PARAMETERS_FILE__NAME: str = "best_hyper_parameters.json"
MODELS_SELECTION__LEARNING_CURVE_DATA__NAME: str = "learning_curve_data.json"
MODELS_SELECTION__SCORES_FILE__NAME: str = "scores.json"
MODELS_SELECTION__CV_RESULTS_FILE__NAME: str = "cv_results.json"
MODELS_SELECTION__BEST_ESTIMATOR_FITTED__NAME: str = "best_estimator_fitted.pkl"


__TUNED_MODELS_FILE_NAMES = [
    MODELS_SELECTION__BEST_HYPER_PARAMETERS_FILE__NAME,
    MODELS_SELECTION__LEARNING_CURVE_DATA__NAME,
    MODELS_SELECTION__SCORES_FILE__NAME,
    MODELS_SELECTION__CV_RESULTS_FILE__NAME,
    MODELS_SELECTION__BEST_ESTIMATOR_FITTED__NAME,
]
TUNED_MODELS_FILE_NAMES = Literal[*__TUNED_MODELS_FILE_NAMES]
TUNED_MODELS_FILE_STEMS = Literal[
    *map(
        lambda file_name: file_name.split(".")[0],
        __TUNED_MODELS_FILE_NAMES,
    )   
]

EXTENSION__JSON: str = "json"
EXTENSION__PKL: str = "pkl"
EXTENSION__PICKLE: str = "pickle"
TUNED_MODELS_FILE_EXTENSIONS = Literal[
    EXTENSION__JSON, 
    EXTENSION__PKL,
    EXTENSION__PICKLE,
]
READ_METHOD__JSON: str = "load_json"
READ_METHOD__PKL: str = "load_pickle"
WRITE_METHOD__JSON: str = "write_json"
WRITE_METHOD__PKL: str = "write_pickle"

TUNED_MODELS_READ_METHODS = Literal[READ_METHOD__JSON, READ_METHOD__PKL]
TUNED_MODELS_WRITE_METHODS = Literal[WRITE_METHOD__JSON, WRITE_METHOD__PKL]

> Créer une classe qui permet d'enregistrer sur ou de lire un fichier bien déterminé (un path bien précis) sur S3. Il faut que cette classe possède : 
> - un attribut qui fournit le path où écrire
> - un connecteur à S3 (qui peut être celui par défaut)
> - une méthode de lecture (car elle dépend de l'extension de l'objet à lire)
> - une méthode d'écriture (idem)
>
> Il faut qu'on puisse lui indiquer _par écrit_ comment il doit lire et écrire, et qu'il se débrouille avec ça pour trouver la bonne méthode pour lecture & écriture

> Faites ça dans un type de classe qui possède un constructeur implicite. Mettez bien du type-hint à chaque fois (sinon vous aurez une fessée)

In [ ]:
class RemoteFileOperator(BaseModel):
    """simple class to read and write pointing to a remote file on S3"""

    ...

> Maintenant codez explicitement deux méthodes pour lire, et pour écrire un objet

In [ ]:
class RemoteFileOperator(BaseModel):
    """simple class to read and write pointing to a remote file on S3"""

    ...

## <a id='toc10_2_'></a>[Héritage, spécialisation, et Factory](#toc0_)

### <a id='toc10_2_1_'></a>[Spécialisation](#toc0_)

On va chercher maintenant à se doter de `RemoteFileOperator` qui soient ***spécialisés*** dans un format de donnée bien défini, ce qui implique de _fixer_ certains attributs à une valeur particulière

> A votre avis, quel est le meilleur cadre théorique pour obtenir cet effet de "spécialisation" d'un objet déjà existant et générique ? 

> Codez 2 classes spécialisées : 
> - 1 pour lire / écrire les fichiers JSON
> - 1 pour lire / écrire les fichiers PKL

In [ ]:
class JsonFileOperator(RemoteFileOperator):
    """declination of RemoteFileOperator in order to operate .json files"""
    ...

> Mettez en pratique votre liseur-écriveur de JSON via un exemple tout simple

In [ ]:
file_name = "test.json"
file_path = f"{TUTORIAL_FOLDER__PATH}/{file_name}"

obj = {"train": -50, "test": -100}

In [ ]:
operator = JsonFileOperator(file_path__remote=file_path)

In [ ]:
operator

In [ ]:
operator.write(obj=obj)

In [ ]:
operator.read()

### <a id='toc10_2_2_'></a>[Factory pour générer des opérateurs spécialisés sans jamais se tromper !](#toc0_)

On souhaite maintenant se doter d'une ___Factory___, qui permettrait de choisir quel est le bon opérateur à appliquer et le former, en fonction du fichier-cible, afin qu'on n'ait, nous, jamais à le faire.

> Concevoir une classe Factory, sans aucun attribut, dont la seule mission c'est de : 
> - constater quel est le type de fichier qu'on lui propose, à partir de son nom (ou de son path)
> - former le bon operateur (parmi JsonFileOperator ou PickleFileOperator) en fonction de l'extension constatée
> - attention à BIEN implémenter de la 'separation of concerns' entre méthodes

In [ ]:
class RemoteFileOperatorFactory(BaseModel):
    """factory class for RemoteFileOperators"""
    ...

> Maintenant gérez proprement le cas où on fournirait un type non-prévu, soit par une Erreur classique, soit par une Erreur custom par vos soins

In [ ]:
class RemoteFileOperatorFactory(BaseModel):
    """factory class for RemoteFileOperators"""
    
    ...

> Optionnel : gérez le type-hint proprement grace au place-holder TypeVar

In [ ]:
RFOpSubClass = TypeVar("RFOpSubClass", bound=RemoteFileOperator)

In [ ]:
class RemoteFileOperatorFactory(BaseModel):
    """factory class for RemoteFileOperators"""
    
    ...

> testez la levée d'erreur en cas de fourniture d'un type non-prévu

In [ ]:
operator = RemoteFileOperatorFactory.from_file_path__remote("tutu.exe")

> testez sur le cas-jouet antérieur

In [ ]:
operator = RemoteFileOperatorFactory.from_file_path__remote(
    file_path__remote=file_path,
)

## <a id='toc10_3_'></a>[Représentants des fichiers à sauvegarder](#toc0_)

On cherche maintenant à représenter par un objet chacun des 4 fichiers d'intérêt que l'on souhaite pouvoir sauvegarder (et lire). 

Les 5 prévus sont : 
- le json de .best_hyper_parameters
- le json de .scores 
- le json de .cv_results
- le pickle de .best_estimator_fitted
- le json de .learning_curve_data

Comme ils sont en type et en nombres déjà connus à l'avance, vous savez ce qui vous attend...

### <a id='toc10_3_1_'></a>[classe représentant un fichier **quelconque**](#toc0_)

On souhaite se doter d'une classe qui représente un fichier remote quelconque.   
Comme cahier des charges de la classe, on a : 
- posséder les attributs suivants : 
  - le nom de fichier, c'est à dire la partie terminale du futur fichier S3, hors du nom du type de régresseur. Par exemple : `"best_hyper_parameters.json"`
  - la racine de ce nom (`stem` de Pathlib, c'est à dire par exemple `"best_hyper_parameters"`)
  - l'extension (par exemple `"json"`)
  - un connecteur vers S3 (de type PolarsDataEngine ou PandasDataEngine)
  - le path S3 vers le **dossier** où chaque fichier futur sera écrit (par exemple `'s3://bkt-pud-uc/at001-quc/td__tuned_model_saver'`)
- posséder un constructeur qui build une instance à partir du seul nom de fichier

Pour s'épargner le `__init__()`, codez ça sous forme de BaseModel

In [ ]:
class TunedModelFile(BaseModel):
    ...

> Maintenant ajoutez-y 2 méthodes pour :
> - construire le remote-path S3 complet du fichier, à partir d'un nom de modèle donné : par exemple si `.file_name = "best_hyper_parameters.json"`, et que le nom du modèle (dont on veut en fait sauvegarder l'attribut `.best_hyper_parameters` sur S3) est `RandomForestRegressor`, alors il faut que cette méthode renvoie `"s3://bkt-pud-uc/.../RandomForestRegressor__best_hyper_parameters.json"`
> - construire l'opérateur adéquate pour lire/écrire à partir d'un path
>
> Metez le bon type-hint ! ;-)

In [ ]:
class TunedModelFile(BaseModel):
    ...

> Testez une instance, qui exploitera la grid formé précédemment, et qui sera en mesure d'écrire ses meilleurs hyper-paramètres sur S3

In [ ]:
tuned_model_file = TunedModelFile.from_file_name(...)

### <a id='toc10_3_2_'></a>[TunedModelFiles déterminés](#toc0_)

On souhaite ne pas former d'instance de `TunedModelFile` à la volée, mais plutôt n'en former que certaines bien précises, au vu du fait qu'il n'y a que quelques types de fichiers bien précis à pouvoir lire / écrire, qui nous intéressent

> A votre avis, quel est le bon "cadre" théorique pour former N instances, N connu d'avance, et itérer dessus ensuite ?

> Former une telle collection des seules instances d'intérêt

In [ ]:
class TunedModelFiles(Enum):
    """enum to operate class TunedModelFile on the specific set of models-files"""

    ...

> Maintenant ajoutez à cette collection : 
> - une méthode pour se représenter de façon plus compacte sous forme de string
> - un "raccourci" pour accéder au stem du TunedModelFile
> - une façon de générer un nom d'attribut explicite qui pointera plus tard vers le chemin dudit fichier sur S3

In [ ]:
class TunedModelFiles(Enum):
    """enum to operate class TunedModelFile on the specific set of models-files"""

    ...

$\Uparrow$ Ce qu'on va souhaiter faire in fine, pour un régresseur déterminé `reg`, c'est, pour chaque membre `tmf` de TunedModelFiles : 
- accéder à l'attribut `tmf.stem` du régresseur (`getattr(reg, tmf.stem)`)
- créer le remote path S3 `tmf.value.build_file_path(model_name=reg.estimator_name)`
- sauvegarder (`getattr(reg, tmf.stem)`) à ce path S3

**Problème de json-ization** : un problème qu'on peut rencontrer, c'est que tout n'est pas "sérialisable" facilement, c'est à dire qu'en pratique on ne peut former de json qu'avec des types simples de la standard library de Python : str, int, float, tuples... et c'est un peu tout. Ainsi, comme `.cv_results` et `.learning_curve_data` contiennent des np.array (qui est un type avancé ne faisant pas partie de la standard library), ils ne sont pas nativement sérialisables : il faut au préalable les transformer en une forme simplifiée pour pouvoir les enregistrer en json

$\Downarrow$ on fournit ici une petite fonction récursive pour rendre un objet json-compliant, afin de transformer les types complexes qui ne peuvent pas être écrits de façon native en json

In [ ]:
def convert_to_json_compliant(value):
    """Converts nested dicts of meta-data to serializable types."""
    if isinstance(value, dict):
        return {k: convert_to_json_compliant(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [convert_to_json_compliant(elem) for elem in value]
    elif isinstance(value, set):
        return [convert_to_json_compliant(elem) for elem in value]
    elif isinstance(value, (np.float32, np.float64, float)):
        if np.isnan(value) or np.isinf(value):
            return None  # or a default value of your choice
        return float(value)
    elif isinstance(value, np.integer):
        return int(value)
    elif isinstance(value, np.ndarray):
        return value.tolist()
    elif isinstance(value, np.ma.MaskedArray):
        return value.filled().tolist()
    elif isinstance(value, Enum):
        return value.name
    else:
        return value

> testez la fonction pour vérifier qu'elle marche bien

> greffez ce code à l'Enum `TunedModelFiles`  
> Quel serait selon vous le moyen le plus naturel de greffer cette fonction à l'Enum ?

In [ ]:
class TunedModelFiles(Enum):
    """enum to operate class TunedModelFile on the specific set of models-files"""
    ...

## <a id='toc10_4_'></a>[Injection du mécanisme de sauvegarde dans la classe `RegressorGridSearcher`](#toc0_)

On va désormais implanter dans la classe `RegressorGridSearcher` un mécanisme de sauvegarde pour agréablement itérer sur l'enum `TunedModelFiles` et s'en servir pour : 
- identifier les "morceaux" de `RegressorGridSearcher` qui nous intéressent
- les associer à des chemins sur S3
- enclencher leur écriture

> Etape 1 : ajoutez à la classe `RegressorGridSearcher` une méthode pour lui implanter, en attributs, les paths s3 de chacun des fichiers que l'on va souhaiter sauvegarder. Cette méthode doit être appelée par le `__init__()` et s'y exécuter

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    """class to perform GridSearchCV and save results on S3"""
    ...

> testez la bonne création de ces attributs sur une instance générée à partir d'un régresseur de votre choix

In [ ]:
grid = RegressorGridSearcher(
    **RANDOM_FOREST_REGRESSOR__FULL_KWARGS,
)

> Etape 2 : Maintenant écrivez pour de bon la méthode de sauvegarde afin que pour chaque objet à écrire sur s3 elle : 
> - se saississe de l'attribut à sauvegarder
> - le transforme en objet json-compliant (cf fonction récursive fournie ci-après)
> - forme son path sur S3
> - l'écrive sur S3
>
> La méthode doit également logger ce qu'elle est en train de faire

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    """class to perform GridSearchCV and save results on S3"""
    
    def __init__(self, **data):
        super().__init__(**data)
        ...

> tester votre super archiveur

In [ ]:
grid = RegressorGridSearcher(
    **RANDOM_FOREST_REGRESSOR__FULL_KWARGS,
)
grid.fit(X=X_train, y=y_train, sample_weight=sample_weights)
grid.compute_score_on_train(X_train, y_train)
grid.compute_score_on_test(X_test, y_test)
grid.compute_learning_curve(X=X, y=y)

In [ ]:
grid.save_on_s3()

> Redéclarez toute l'Enum `RegressorGridSearchers` pour la mettre à jour des nouveaux `RegressorGridSearcher`, itérez et sauvegardez tout en un minimum de lignes de comande, très claires pour un futur repreneur de votre code !!!

In [ ]:
class RegressorGridSearchers(Enum):
    """simple Enum to describe all RegressorGridSearcher formed"""
    
    RANDOM_FOREST_REGRESSOR = RegressorGridSearcher(
        **RANDOM_FOREST_REGRESSOR__FULL_KWARGS
    )
    
    ELASTIC_NET = RegressorGridSearcher(
        **ELASTIC_NET__FULL_KWARGS,
    )
    
    K_NEIGHBORS_REGRESSOR = RegressorGridSearcher(
        **K_NEIGHBORS_REGRESSOR__FULL_KWARGS,
    )

In [ ]:
for grid_searcher in RegressorGridSearchers:
    logger.info(f"\n\nRegressorGridSearcher : {grid_searcher.name}")
    grid = grid_searcher.value 
    logger.info(f"training regressor {grid.estimator_name} on X_train & y_train")
    grid.fit(X=X_train, y=y_train, sample_weight=sample_weights)
    logger.success(f"[OK] fit of regressor {grid.estimator_name} successful")
    grid.compute_score_on_test(X_test, y_test)
    grid.compute_learning_curve(X=X, y=y)
    logger.info(f"scores reached : {grid.scores}")
    logger.info(f"best hyper-parameters identified : {grid.best_hyper_parameters}")
    logger.info("saving on S3")
    grid.save_on_s3()

## <a id='toc10_5_'></a>[Injection d'un mécanisme de _lecture_](#toc0_)

On va cette fois-ci se doter d'un mécanisme de _lecture_ des résultats enregistrés sur S3, afin de pouvoir régénérer une instance _entraînée_ de `RegressorGridSearcher` pour peu qu'on aille lire ce qui la concerne sur S3, qu'on avait préalablement sauvegardé

> Etape 1 : dotez désormais votre classe `RegressorGridSearcher` d'un mécanisme de _lecture_, tout à fait symétrique à celui d'écriture (codé en gros de la même façon)

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(
        self, 
        accepts_sample_weight: bool,
        **data,
    ):
        super().__init__(**data)
        self.accepts_sample_weight = accepts_sample_weight
        ...

> Remettez `RegressorGridSearcher` dans son Enum `RegressorGridSearchers`, saisissez-vous d'un membre, tentez une opération de lecture, et constatez que ça plante.  

> A votre avis, pourquoi ?

In [ ]:
class RegressorGridSearchers(Enum):
    """simple Enum to describe all RegressorGridSearcher formed"""
    ...

> Proposez une façon "batarde" de contourner le problème, en éliminant les properties (sur lesquelles vous n'avez pas le droit d'écrire), et en les remplaçant par de simples _attributs_, créés et affectés pendant la phase de `__init__()`.  
> Attention à gérer intelligemment le cas où certains attributs de `GridSearchCV` n'existent pas encore avant qu'un fit n'ait eu lieu !

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(
        self, 
        accepts_sample_weight: bool,
        **data,
    ):
        super().__init__(**data)
        self.accepts_sample_weight = accepts_sample_weight
        ...

In [ ]:
class RegressorGridSearchers(Enum):
    """simple Enum to describe all RegressorGridSearcher formed"""
    ...

In [ ]:
rgs = RegressorGridSearchers.RANDOM_FOREST_REGRESSOR.value
rgs.read_from_s3()

> Montrez en quoi ça n'est pas une solution parfaitement satisfaisante (quels attributs ne se voient jamais affecter leur "vraie" valeur ?)

> procédez maintenant à la vraie modification pythonic, avec les setters

In [ ]:
class RegressorGridSearcher(GridSearchCV):
    
    def __init__(
        self, 
        accepts_sample_weight: bool,
        **data,
    ):
        super().__init__(**data)
        self.accepts_sample_weight = accepts_sample_weight
        
        # * supplemental info on (X_train, y_train) & (X_test, y_test)
        self.score__train = None
        self.score__test = None
        
        # * learning curve
        self.train_sizes = None 
        self.train_scores = None
        self.test_scores = None
        self.learning_curve_displayer = None

	    # * creation of paths for future parts save
        self._set_paths()

    @property 
    def estimator_name(self) -> str:
        return self.estimator.__class__.__name__
    
    @property 
    def scores(self) -> dict[str, float]:
        return dict(
            train=self.score__train,
            test=self.score__test,
        )
    
    @scores.setter
    def scores(
        self, 
        scores: dict[str, float],
    ) -> None:
        self.score__train = scores.get("train")
        self.score__test = scores.get("test")

    @property 
    def learning_curve_data(self) -> dict[str, int|float]:
        return dict(
            train_sizes=self.train_sizes,
            train_scores=self.train_scores,
            test_scores=self.test_scores,
        )
    
    @learning_curve_data.setter
    def learning_curve_data(
        self,
        learning_curve_data: dict,
    ) -> None:
        self.train_sizes = np.array(learning_curve_data.get("train_sizes"))
        self.train_scores = np.array(learning_curve_data.get("train_scores"))
        self.test_scores = np.array(learning_curve_data.get("test_scores"))
    
    @property
    def best_hyper_parameters(self) -> dict[str, Any]:
        return self.best_params_
    
    @best_hyper_parameters.setter
    def best_hyper_parameters(
        self,
        best_params_: dict[str, Any],
    ) -> None:
        self.best_params_ = best_params_
    
    @property 
    def cv_results(self) -> dict:
        return self.cv_results_
    
    @cv_results.setter
    def cv_results(
        self,
        cv_results_: dict,
    ) -> None:
        self.cv_results_ = cv_results_
    
    @property 
    def best_estimator_fitted(self) -> Any:
        return self.best_estimator_
    
    @best_estimator_fitted.setter
    def best_estimator_fitted(
        self,
        best_estimator_: Any,
    ) -> None:
        self.best_estimator_ = best_estimator_


    def _set_paths(
        self,
    ) -> None:
        for tuned_model_file in TunedModelFiles:
            path_name = tuned_model_file.path_name 
            path = tuned_model_file.value.build_file_path(
                model_name=self.estimator_name,
            )
            setattr(self, path_name, path)

    def fit(
        self,
        X,
        y,
        sample_weight: pd.Series | np.ndarray | None = None,
        **kwargs,
    ) -> None:
        fit_kwargs = dict(
            X=X,
            y=y,
        )
        if self.accepts_sample_weight and sample_weight is not None:
            logger.info(f"since estimator {self.estimator_name} is compatible with the use of sample-weight, sample_weight is used during fit")
            fit_kwargs["sample_weight"] = sample_weight
        fit_kwargs.update(kwargs)
        super().fit(**fit_kwargs)
        self.compute_score_on_train(
            X_train=X,
            y_train=y,
        )

    def _compute_score(
        self,
        X: np.ndarray,
        y: np.ndarray,
    ) -> float:
        score = self.score(
            X=X,
            y=y,
        )
        
        return score

    def compute_score_on_train(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
    ) -> float:
        score__train = self._compute_score(
            X=X_train,
            y=y_train,
        )
        self.score__train = score__train
        return score__train
    
    def compute_score_on_test(
        self,
        X_test: np.ndarray,
        y_test: np.ndarray,
    ) -> float:
        score__test = self._compute_score(
            X=X_test,
            y=y_test,
        )
        self.score__test = score__test
        return score__test
    
    def compute_learning_curve(
        self,
        X: np.ndarray,
        y: np.ndarray,
        train_sizes: list[float] | None = None,
        *args,
        **kwarg,
    ) -> None:
        if train_sizes is None:
            train_sizes = LEARNING_CURVE__TRAIN_SIZES
        if not kwarg:
            kwargs = REGRESSOR_GRID_SEARCHER__COMMON_KWARGS
        train_sizes, train_scores, test_scores = learning_curve(
            estimator=deepcopy(self.best_estimator_),
            X=X,
            y=y,
            train_sizes=train_sizes,
            *args,
            **kwargs,
        )
        self.train_sizes = train_sizes
        self.train_scores = train_scores
        self.test_scores = test_scores
        
        return (
            self.train_sizes,
            self.train_scores,
            self.test_scores,
        )
    
    def display_learning_curve(self) -> None:
        if self.learning_curve_displayer is None:
            self.learning_curve_displayer = LearningCurveDisplay(**self.learning_curve_data)
        self.learning_curve_displayer.plot(score_name="score (neg. RMSE)")
        plt.show()
    
    def save_on_s3(self):
        for tuned_model_file in TunedModelFiles:
            obj = getattr(self, tuned_model_file.stem)
            obj__compliant = TunedModelFiles.make_json_compliant(obj)
            obj__path = tuned_model_file.value.build_file_path(model_name=self.estimator_name)
            logger.info(f"about to write {tuned_model_file.stem} @ {obj__path}")
            operator = tuned_model_file.value.build_operator(model_name=self.estimator_name)
            operator.write(obj=obj__compliant)
            logger.success(f"[OK] {tuned_model_file.stem} successfully saved on s3 @ {obj__path}")

    def read_from_s3(self):
        for tuned_model_file in TunedModelFiles:
            obj__attribute_name = tuned_model_file.stem
            obj__path = tuned_model_file.value.build_file_path(model_name=self.estimator_name)
            logger.info(f"about to read and load object saved son s3 @ {obj__path}")
            operator = tuned_model_file.value.build_operator(model_name=self.estimator_name)
            obj = operator.read()
            setattr(self, obj__attribute_name, obj)
            logger.success(f"[OK], obj loaded and set at .{obj__attribute_name}")


    def __str__(self) -> str:
        return f"{self.__class__.__name__}(estimator={self.estimator_name})"